# GPU Assignment II — HW4.a + HW4.5 (ONE GPU)

**RTX 4090 / 24 GB.** Professor Savitha V clarified that one GPU is sufficient and Part C is optional. This notebook runs the single-GPU experiments. Use the **same original Twitter CSV, SLICE, SEED, and split** throughout. Full-run requirements not implemented in this starter are marked as such. Run from repository root.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json
# RunPod: open JupyterLab in gpu-assignment-ii, or set PROJECT_ROOT explicitly.
ROOT=Path(os.environ.get('PROJECT_ROOT',Path.cwd())).resolve()
if not (ROOT/'configs/experiment.yaml').exists():
    matches=list(Path.cwd().rglob('configs/experiment.yaml'))
    assert matches, 'Cannot locate configs/experiment.yaml; set PROJECT_ROOT to extracted project folder'
    ROOT=matches[0].parent.parent
os.chdir(ROOT)
print('Project root:',ROOT)
subprocess.run(['nvidia-smi','-L'],check=True)
import torch
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU required; any single supported GPU is allowed'
print('GPU:',torch.cuda.get_device_name(0))


## 1 — Dataset and reproducibility
Provide `data/twitter.csv`, set `slice` and `seed` in YAML. Exact row range is written to `results/row_range.json`. 70/15/15 stratified split.

In [ ]:
sys.path.insert(0,str(ROOT/'src'))
from common import config,load_data
c=config()
tr,va,te,label_map=load_data(c)
print('Rows:',len(tr),len(va),len(te),'Classes:',label_map)
print((ROOT/'results/row_range.json').read_text())

## 2 — HW4.a BERT fine-tuning
Train baseline and capture epoch-by-epoch loss, validation accuracy, peak GPU allocation, wall time, and best validation epoch.

In [ ]:
subprocess.run([sys.executable,'src/train_bert.py'],check=True)

## 3 — HW4.a test evaluation
Confusion matrix, per-class precision/recall, worst-class examples. **Write one-line diagnoses yourself** in the generated JSON; the script cannot determine qualitative causes.

In [ ]:
subprocess.run([sys.executable,'src/evaluate_bert.py'],check=True)
from IPython.display import display,Image
display(Image(filename='figures/training_curves.png'))
display(Image(filename='figures/confusion_matrix.png'))
print((ROOT/'results/worst_class_errors.json').read_text()[:5000])

## 4 — HW4.a serving baseline

The serving model is a separate causal LM (BERT classification cannot run in vLLM generation mode). Report **batch-completion p50/p95** as such; they are not true independent request latencies or TTFT. A concurrent client is required for true request-level latency. Increase repeats for stable percentiles.


In [ ]:
for backend,b in [('hf',1),('vllm',1),('vllm',8),('vllm',32)]:
    subprocess.run([sys.executable,'src/benchmark_vllm.py','--backend',backend,'--batch',str(b)],check=True)
import pandas as pd
display(pd.read_csv('results/serving_metrics.csv'))

## 5 — Part A: measured full-epoch batch scaling
Run successive batch sizes until OOM. Use measured samples/sec, peak allocated GPU memory and epoch seconds. Keep the same dataset, seed and training settings.


In [ ]:
subprocess.run([sys.executable,'src/full_experiments.py','--mode','scaling','--max-batch','1024'],check=True)
display(pd.read_csv('results/full_batch_scaling.csv'))


## 6 — Part B: measured independent techniques and data-selected combination

Run baseline, AMP, checkpointing, gradient accumulation and 8-bit optimizer **first**. The script then selects eligible techniques based on observed memory, throughput and accuracy, writes `results/combination_selection.json`, and runs that combination. Compare accumulation separately because it changes effective batch. Selection thresholds are heuristics and must be justified in your report.


In [ ]:
# Full-epoch, independently measured techniques. Scans max batch until OOM (can take time).
subprocess.run([sys.executable,'src/full_experiments.py','--mode','techniques','--scan-technique-max','--max-batch','1024'],check=True)
display(pd.read_csv('results/full_memory_comparison.csv'))


## 7 — Plot measured single-card throughput and memory
Use only successful observed probes; do not treat estimated full-epoch seconds as measured.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
Path('figures').mkdir(exist_ok=True)
a=pd.read_csv('results/full_batch_scaling.csv');a=a[a.status=='ok']
fig,ax=plt.subplots();ax.plot(a.batch,a.samples_per_second,'o-');ax.set(xlabel='Batch size',ylabel='Samples / sec',title='Measured full-epoch batch throughput');fig.savefig('figures/throughput_vs_batch.png',bbox_inches='tight');plt.show()
b=pd.read_csv('results/full_memory_comparison.csv');b=b[(b.status=='ok') & (b.batch==int(c['batch_size']))].drop_duplicates('technique',keep='last')
fig,ax=plt.subplots(figsize=(9,4));ax.bar(b.technique,b.peak_gb);ax.tick_params(axis='x',rotation=30);ax.set(ylabel='Peak allocated GB',title='Measured full-epoch technique memory');fig.tight_layout();fig.savefig('figures/memory_comparison.png');plt.show()


## 8 — Part C: optional, skipped

The instructor explicitly made Part C optional. This notebook does not launch distributed training. `METRICS.md` includes the required exact statement: **Part C skipped (optional)**.


In [ ]:
print('Part C skipped (optional)')


## 9 — Part D: vLLM context/KV-cache sweep on one GPU

The script starts a **new vLLM engine** at increasing `max_model_len` values with batch 8, recording success/failure and full startup logs (including GPU-block messages when emitted by the installed vLLM version). It also records **analytical KV bytes/token**; `nvidia-smi` is vLLM's reserved process memory, **not** measured KV consumption. If the model's supported context limit is reached before an engine failure, say so; do not claim GPU exhaustion.


In [ ]:
subprocess.run([sys.executable,'src/kv_context_sweep.py'],check=True)
display(pd.read_csv('results/kv_context_sweep.csv'))
print((ROOT/'results/kv_context_summary.json').read_text())


## 10 — Part E: summary, RunPod evidence, and GPU-hours

Generate the measured GPU-hours log and preserve `nvidia-smi -L` output. Fill in qualitative written answers and record the instructor's optional-Part-C clarification. Use actual GPU identity, not a hard-coded RTX 4090 label.


In [ ]:
subprocess.run([sys.executable,'src/finalize_evidence.py'],check=True)
subprocess.run([sys.executable,'src/summarize_results.py'],check=False)
print((ROOT/'gpu_hours.csv').read_text())
print((ROOT/'GPU_INFO.txt').read_text()[:1500])


## 11 — Final checklist

- [ ] Original HW4.a Twitter CSV, unchanged SLICE and SEED, exact row range documented
- [ ] `GPU_INFO.txt` contains actual `nvidia-smi -L` output or screenshot saved
- [ ] HW4.a training/validation/testing, worst-class error diagnoses
- [ ] HF and vLLM throughput, p50/p95 latency, written analysis
- [ ] Part A measured full-epoch scaling, OOM boundary (one GPU)
- [ ] Part B independent techniques, maximum batch, full-run accuracy and throughput, ranked table
- [ ] Part C skipped (optional) written in METRICS.md
- [ ] Part D KV-cache memory and longest context at batch 8, one GPU
- [ ] Part E GPU-hours, best configuration, plots and complete measured results
- [ ] RUN_LOG.txt, AI_USE.md, applicable captures and records, Git tag hw4-5


## 12 — Submission audit and theory notes
Run the audit script after experiments. It reports missing outputs; it does not fabricate results. Read `SUBMISSION_AUDIT.md` and `THEORY_VIVA_GUIDE.md`.

In [ ]:
subprocess.run([sys.executable,'src/submission_audit.py'],check=False)
print((ROOT/'results/audit_report.txt').read_text())
